<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>03. 🌦️ Weather Forecast API Consumer</b>
</div>


### 📌 Project Overview
In this project, we create an interactive weather query application that retrieves real-time local conditions and multi-day forecasts using the free, keyless **wttr.in API**.
Students learn how to consume external JSON APIs, handle query parameters, format outputs, and design robust error handling wrappers.

#### 🔑 Key Concepts Covered:
- Performing requests with query parameters using `requests`
- Error propagation via `response.raise_for_status()`
- Graceful handling of network timeouts, connection drops, and API failures
- JSON extraction and structured CLI reporting


In [ ]:
import requests
import json

def fetch_weather(city):
    """Query wttr.in for real-time and daily forecast data."""
    # URL encode city name by replacing spaces with +
    url_city = city.replace(' ', '+')
    api_url = f'https://wttr.in/{url_city}'
    parameters = {
        'format': 'j1'
    }
    
    try:
        response = requests.get(api_url, params=parameters, timeout=8)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.Timeout:
        print('❌ Connection timed out. Please try again.')
    except requests.exceptions.HTTPError as e:
        print(f'❌ Server error ({response.status_code}): {e}')
    except requests.exceptions.RequestException as e:
        print(f'❌ Network or connection failure: {e}')
    return None
    
def generate_report(data):
    if not data:
        return
        
    current = data.get('current_condition', [{}])[0]
    weather_list = data.get('weather', [])
    area_info = data.get('nearest_area', [{}])[0]
    city_name = area_info.get('areaName', [{}])[0].get('value', 'Unknown Location')
    country_name = area_info.get('country', [{}])[0].get('value', 'Unknown Country')
    
    print('=' * 55)
    print(f'🌦️ WEATHER REPORT - {city_name}, {country_name}')
    print('=' * 55)
    print(f'Current Temp : {current.get("temp_C")}°C (Feels like {current.get("FeelsLikeC")}°C)')
    print(f'Wind Speed   : {current.get("windspeedKmph")} km/h ({current.get("winddir16Point")})')
    print(f'Condition    : {current.get("weatherDesc", [{}])[0].get("value")}')
    print(f'Last Updated : {current.get("observation_time")}')
    print('-' * 55)
    print('📅 3-Day Outlook:')
    print('-' * 55)
    print(f'{"Date":<12} | {"Min Temp":<8} | {"Max Temp":<8} | {"Rain (mm)":<8}')
    print('-' * 55)
    
    for day in weather_list:
        date_str = day.get('date')
        t_min = day.get('mintempC')
        t_max = day.get('maxtempC')
        precip_sum = sum(float(hour.get('precipMM', 0)) for hour in day.get('hourly', []))
        print(f'{date_str:<12} | {t_min:>6}°C | {t_max:>6}°C | {precip_sum:>7.1f} mm')
    print('=' * 55)


In [ ]:
# Query New Delhi weather
city_input = 'New Delhi'
print(f'Querying forecast for {city_input}...')
weather_data = fetch_weather(city_input)
generate_report(weather_data)
